In [ ]:
import joblib
import random
import optuna
from nn_models   import *
from nn_data     import *
from nn_training import *
from nn_score_and_plot import *
from torch.utils.data            import DataLoader
from sklearn.preprocessing       import StandardScaler
import matplotlib.pyplot as plt

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
DATA_PATH = '/path'
OUT_PATH = '/path'

In [ ]:
features = KEY_FEATURES + CONTROL_FEATURES
df_train = pd.read_csv(DATA_PATH + 'train.csv')
df_val   = pd.read_csv(DATA_PATH + 'val.csv')

print(f'Train samples  : {len(df_train):,}')
print(f'Val samples    : {len(df_val):,}')

In [ ]:
def make_objective(df_tr, df_vl, features, n_sources, n_fkey, n_fcontrol):
    """
    Returns an Optuna objective that searches over architecture and training
    hyperparameters using spatial 5-fold CV.
    Folds are split by monitoring station (stcode) to avoid spatial leakage.
    Scalers are re-fitted inside every fold.
    """
    def objective(trial):
        
        # Search space
        h1, h2, hout = suggest_chain(trial, ["h1", "h2", "hout"], low=4, high=7) 
        hh1, hh2     = suggest_chain(trial, ["hh1", "hh2"], low=3, high=5)
        hsk          = trial.suggest_categorical("hsk",  [8, 16, 32])
        drop_rate    = trial.suggest_float("drop_rate",  0.0, 0.3, step=0.1)
        hdrop_rate   = trial.suggest_float("hdrop_rate", 0.0, 0.3, step=0.1)
        lr           = trial.suggest_float("lr",         1e-4, 1e-2, log=True)
        lambda_sim   = trial.suggest_float("lambda_sim", 1e-2, 5e-2, step=0.01)

        df_tr_work = df_tr.copy()
        df_vl_work = df_vl.copy()
        
        # Fit scalers inside fold — no leakage
        f_sc = StandardScaler()
        t_sc = StandardScaler()
        df_tr_work[features]      = f_sc.fit_transform(df_tr_work[features])
        df_tr_work['pm25_filled'] = t_sc.fit_transform(df_tr_work['pm25_filled'].values.reshape(-1, 1))
        df_vl_work[features]      = f_sc.transform(df_vl_work[features])
        df_vl_work['pm25']        = transform_target_preserve_nan(df_vl_work['pm25'], t_sc)

        tr_groups = prepare_3d_input(df=df_tr_work, ts_cols=features, target_col='pm25_filled', verbose=False)
        vl_groups = prepare_3d_input(df=df_vl_work, ts_cols=features, verbose=False)

        tr_tensors = groups_to_tensors(tr_groups, mode='cnn1d', reverse_sort=True)
        vl_tensors = groups_to_tensors(vl_groups, mode='cnn1d')

        physical_zero_std = -f_sc.mean_[:n_sources] / f_sc.scale_[:n_sources]

        # Train
        m = TwoStageCNN1D(
                n_sources  = n_sources,                            
                n_fkey     = n_fkey,
                n_fcontrol = n_fcontrol,               
                h1         = h1, 
                h2         = h2, 
                hout       = hout, 
                drop_rate  = drop_rate,         
                hh1        = hh1, 
                hh2        = hh2, 
                hdrop_rate = hdrop_rate, 
                hsk        = hsk,                  
                threshold  = physical_zero_std,
                **source_sum_scaler_params(f_sc, t_sc, n_sources)) 
        opt  = torch.optim.Adam(m.parameters(), lr=lr)
        
        train_losses, best_val, best_epoch, val_losses = train_TwostageCNN1D(
            m, tr_tensors, opt, MAX_EPOCHS, lambda_sim=lambda_sim,
            vl_tensors=vl_tensors, patience=PATIENCE)

        n_features = n_fkey + n_fcontrol
        train_pred_df = collect_TwoStageCNN1D_preds(tr_groups, df_tr, m, t_sc, f_sc, 
                                                    CMAQ_FEATURES, n_features, target_col='pm25_filled')
        val_pred_df   = collect_TwoStageCNN1D_preds(vl_groups, df_vl, m, t_sc, f_sc, 
                                                    CMAQ_FEATURES, n_features)
        train_score = calculate_score(obs=train_pred_df['pm25_filled'].values, pred=train_pred_df['pred'].values)
        val_pred_eval = val_pred_df.dropna(subset=['pm25']) 
        val_score   = calculate_score(obs=val_pred_eval['pm25'].values, pred=val_pred_eval['pred'].values)
        
        trial.set_user_attr('train_score', train_score.tolist())
        trial.set_user_attr('val_score',   val_score.tolist())
        trial.set_user_attr("h1", h1)
        trial.set_user_attr("h2", h2)
        trial.set_user_attr("hout", hout)
        trial.set_user_attr("hh1", hh1)
        trial.set_user_attr("hh2", hh2)
        trial.set_user_attr('best_epoch',  best_epoch)
        trial.set_user_attr('train_losses', train_losses)
        trial.set_user_attr('val_losses',   val_losses)

        print(f"  trial {trial.number}  best_val {best_val:.4f}  best_epoch {best_epoch}",
              flush=True)

        return float(best_val)

    return objective

def optuna_callback(study, trial):
    if trial.state != optuna.trial.TrialState.COMPLETE:
        return
    done = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    tag  = "  ← new best" if trial.value == study.best_value else ""
    best_epoch  = trial.user_attrs.get('best_epoch', '?')
    print(f"[{done:3d}/{N_TRIALS}]  trial {trial.number:3d} "f"Val MSE {trial.value:.4f}  |  best {study.best_value:.4f} "f"best_epoch {best_epoch}{tag}")

In [ ]:
seed_everything()
N_TRIALS   = 25
TIMEOUT    = None   

# original
MAX_EPOCHS  = 100    
PATIENCE    = 10

study = optuna.create_study(
    direction  = "minimize",
    sampler    = optuna.samplers.RandomSampler(seed=88),
    study_name = "twostage_cnn1d")

study.optimize(
    make_objective(df_train, df_val, features, N_SOURCES, N_FKEY, N_FCONTROL),
    n_trials  = N_TRIALS,
    timeout   = TIMEOUT,
    callbacks = [optuna_callback])

In [ ]:
best_trial   = study.best_trial
train_curves = best_trial.user_attrs['train_losses']   
val_curves   = best_trial.user_attrs['val_losses']    

fig, ax = plt.subplots(figsize=(6, 4))
fz=16
ax.plot(train_curves, label='Train')
ax.plot(val_curves,   label='Validation')
ax.axvline(best_trial.user_attrs['best_epoch']- 1,
           color='red', linestyle='--', label='Best epoch')
ax.set_title(f'Two-stage CNN1D', fontsize=fz)
ax.set_xlabel('Epoch', fontsize=fz)
ax.tick_params(axis='both', labelsize=fz)
ax.grid(True, linestyle='dotted')
ax.legend(fontsize=fz, loc='upper right')
ax.set_ylabel('Two-stage Loss', fontsize=fz)

In [ ]:
print(f"Best CV loss : {study.best_value:.6f}")
print(f"Best params  : {study.best_params}")
print(f"Best epoch   : {study.best_trial.user_attrs['best_epoch']}")
print(f"h1   : {study.best_trial.user_attrs['h1']}")
print(f"h2   : {study.best_trial.user_attrs['h2']}")
print(f"hout : {study.best_trial.user_attrs['hout']}")
print(f"hh1  : {study.best_trial.user_attrs['hh1']}")
print(f"hh2  : {study.best_trial.user_attrs['hh2']}")
print(f"train score  : {study.best_trial.user_attrs['train_score']}")
print(f"val score    : {study.best_trial.user_attrs['val_score']}")

In [ ]:
# joblib.dump(study, OUT_PATH)